# 获取并发送 SSH 公钥

请在**你自己的电脑**上运行本 Notebook。它只查找 `.pub` 公钥文件，不读取私钥内容，也不会自动上传或连接云实例。

运行最后一个代码单元后，如果提示已复制，请把剪贴板中的**完整公钥行**发送给 Alex，由 Alex 按队内流程添加到共享实例。若没有可用公钥，先用本机终端生成一对专用密钥，再运行单元。

Windows PowerShell：`$key = "$env:USERPROFILE\.ssh\amd_cloud_team"; ssh-keygen -t ed25519 -C "your-name@team" -f $key`

macOS / Linux：`mkdir -p ~/.ssh && chmod 700 ~/.ssh && ssh-keygen -t ed25519 -C "your-name@team" -f ~/.ssh/amd_cloud_team`

`ssh-keygen` 会生成 `.pub` 公钥和无后缀私钥。只把 `.pub` 的内容发给 Alex；**绝不要发送、上传或提交无后缀私钥文件**。如果设置了密钥口令，不要把口令发给任何人。

In [ ]:
from pathlib import Path
import shutil
import subprocess

SSH_DIR = Path.home() / ".ssh"
public_key_files = sorted(path for path in SSH_DIR.glob("*.pub") if path.is_file())

if not public_key_files:
    SELECTED_PUBLIC_KEY = None
    print("没有找到 .pub 公钥文件。请先按上方命令生成专用密钥，再重新运行此单元。")
else:
    preferred = [SSH_DIR / "amd_cloud_team.pub", SSH_DIR / "id_ed25519.pub"]
    SELECTED_PUBLIC_KEY = next(
        (path for path in preferred if path in public_key_files), public_key_files[0]
    )
    ssh_keygen = shutil.which("ssh-keygen")
    print("本机找到的公钥：")
    for path in public_key_files:
        fingerprint = "指纹工具不可用"
        if ssh_keygen:
            result = subprocess.run(
                [ssh_keygen, "-lf", str(path)], capture_output=True, text=True, check=False
            )
            if result.returncode == 0:
                fields = result.stdout.split()
                fingerprint = " ".join(fields[:2]) if len(fields) >= 2 else "指纹读取失败"
        marker = "（默认选择）" if path == SELECTED_PUBLIC_KEY else ""
        print(f"- {path.name}: {fingerprint} {marker}")
    print("如要使用其他公钥，请在下一单元修改 SELECTED_PUBLIC_KEY。")

In [ ]:
import os
import sys

if SELECTED_PUBLIC_KEY is None:
    print("尚无公钥可复制。生成密钥后，请从第一个代码单元重新运行。")
else:
    public_key = SELECTED_PUBLIC_KEY.read_text(encoding="utf-8").strip().splitlines()[0]
    fields = public_key.split()
    valid_prefixes = (
        "ssh-ed25519", "ssh-rsa", "ecdsa-sha2-", "sk-ssh-ed25519@", "sk-ecdsa-sha2-"
    )
    if len(fields) < 2 or not fields[0].startswith(valid_prefixes):
        print("所选文件内容不像有效的 OpenSSH 公钥；请确认选择的是 .pub 文件。")
    else:
        copied = False
        if os.name == "nt":
            clipboard_commands = [["clip"]]
        elif sys.platform == "darwin":
            clipboard_commands = [["pbcopy"]]
        else:
            clipboard_commands = [
                ["wl-copy"],
                ["xclip", "-selection", "clipboard"],
                ["xsel", "--clipboard", "--input"],
            ]

        for command in clipboard_commands:
            if shutil.which(command[0]):
                try:
                    result = subprocess.run(
                        command, input=public_key + "\n", text=True, capture_output=True, check=False
                    )
                    if result.returncode == 0:
                        copied = True
                        break
                except OSError:
                    pass

        if copied:
            print(f"已将 {SELECTED_PUBLIC_KEY.name} 的公钥复制到剪贴板。请粘贴并发送给 Alex。")
            print("只发送公钥；不要发送对应的无后缀私钥文件。")
        else:
            print("无法访问系统剪贴板，请复制下面这一行并发送给 Alex：")
            print(public_key)